# 面试问题：LLM Serving 的 CUDA Graph capture/replay、BatchDescriptor 与三级 dispatch 怎样设计？

**一句话回答。** 不要把 CUDA Graph 理解成“打开一个加速开关”，而要把它设计成受约束的运行时状态机：调度器先把真实请求归一化为 `BatchDescriptor`，再查询已批准的 capture key，按 `FULL → PIECEWISE → NONE(eager)` 的优先级选择路径；wrapper 只负责为静态地址和兼容 shape 执行 capture/replay。动态 shape、LoRA、后端能力或缓冲容量不满足合同时必须可观测地降级，不能硬 replay。

本 Notebook 不需要真实 GPU，只用 Python 小状态机模拟 capture、replay、静态缓冲、动态 shape fallback 和验收指标。它证明的是控制面合同与失败分支，不代表真实 CUDA 内核性能。

**资料入口。** [vLLM CUDA Graphs 设计文档](https://docs.vllm.ai/en/latest/design/cuda_graphs/) 说明了运行模式、`BatchDescriptor`、集中式 dispatcher、wrapper 与后端兼容性；本文按这些概念做教学化简。

In [ ]:
question = "怎样设计 LLM Serving 的 CUDA Graph 调度状态机？"  # 定义本 Notebook 要回答的核心面试问题。
runtime_modes = ("FULL", "PIECEWISE", "NONE")  # 声明调度器最终允许返回的三个具体运行模式。
assert "CUDA Graph" in question  # 校验问题确实聚焦 CUDA Graph 调度主题。
assert runtime_modes[0] == "FULL"  # 校验全图模式拥有最高候选优先级。
assert runtime_modes[-1] == "NONE"  # 校验无图模式作为 eager 回退出口。

## 1. 先讲清 capture 与 replay 的合同

普通 eager 每轮都由 CPU 发起一串 kernel；CUDA Graph capture 把一条兼容执行路径及其依赖记录下来，replay 时减少重复提交开销。代价是捕获期看到的地址、控制流和 shape 必须满足复用合同，所以服务端常预先选择有限的 capture size，将请求 padding 到最近桶。桶不是越多越好：更多桶会增加捕获时间、图对象与静态缓冲显存；桶太少又会增加 padding 或 eager 回退。

面试中应强调：graph 优化的是调度开销，不改变模型数学定义。任何图路径都要和 eager reference 做数值或语义对齐。

In [ ]:
capture_sizes = (4, 8, 16)  # 配置允许预热或捕获的 token 总数桶。
def pad_token_count(token_count):  # 定义把动态 token 数归一化到 capture 桶的函数。
    return next((size for size in capture_sizes if size >= token_count), None)  # 返回最小可容纳桶，超上限时返回空值触发 eager。
assert pad_token_count(3) == 4  # 校验小请求被补齐到四 token 桶。
assert pad_token_count(5) == 8  # 校验非精确 shape 被补齐到最近更大桶。
assert pad_token_count(17) is None  # 校验超出最大捕获桶的请求不会冒险复用图。

## 2. BatchDescriptor 是唯一的批次身份

只拿 batch size 当 key 不够，因为相同请求数可能有不同 token 总数、query length 分布、LoRA 状态和注意力路径。`BatchDescriptor` 应由执行器而不是 LLM 生成，并携带调度真正需要的最小字段：padding 后 token 数、请求数、是否 uniform、是否有额外适配器，以及本教学例中的 cascade 标志。

`uniform=True` 表示各请求 query length 一致，不能偷换成“每个请求一定只有一个 token”；speculative decode 的验证批也可能是统一的多 token query。描述符一旦参与选图，就应在一次 forward context 内保持不可变。

In [ ]:
def make_descriptor(query_lengths, has_lora=False, cascade=False):  # 根据请求长度和兼容性标志构造教学版批次描述符。
    total_tokens = sum(query_lengths)  # 汇总本轮实际输入 token 数以便选择 padding 桶。
    return {"num_tokens": pad_token_count(total_tokens), "actual_tokens": total_tokens, "num_reqs": len(query_lengths), "uniform": len(set(query_lengths)) == 1, "has_lora": has_lora, "cascade": cascade}  # 返回调度所需的最小且确定性字段。
decode_batch = make_descriptor([1, 1, 1, 1])  # 构造四请求的统一单 token decode 批次。
mixed_batch = make_descriptor([3, 2])  # 构造 query length 不一致的混合或 prefill 批次。
oversize_batch = make_descriptor([9, 9])  # 构造超过最大 capture size 的动态大批次。
assert decode_batch["num_tokens"] == 4 and decode_batch["uniform"]  # 校验 decode 批次落入四 token 全图候选桶。
assert mixed_batch["num_tokens"] == 8 and not mixed_batch["uniform"]  # 校验混合批次补齐到八 token 且不是 uniform。
assert oversize_batch["num_tokens"] is None  # 校验超大批次无法形成已批准的 capture key。
assert decode_batch["num_reqs"] == 4  # 校验描述符保留请求数而不是只记录 token 数。

## 3. Dispatcher 是可用 key 的唯一事实源

集中式 dispatcher 维护 FULL 与 PIECEWISE 的合法 key 集合，wrapper 不应各自猜测 shape 是否存在。默认双模式下，uniform decode 优先命中 FULL；其他兼容批次尝试 PIECEWISE；没有合法 key 就返回 NONE。cascade attention 在设计文档中不能走完整图，因此应优先路由 piecewise，若 piecewise 不可用才 eager。

这里把 LoRA 设为教学系统尚未捕获的组合，所以强制 NONE。生产实现可以捕获更多组合，但必须让 `has_lora` 等会改变 kernel/权重地址的维度进入 key，绝不能静默复用不相容图。

In [ ]:
full_keys = {(4, 4, True, False), (8, 8, True, False)}  # 登记允许 FULL capture 的描述符 key 集合。
piecewise_keys = {(4, False), (8, False), (16, False)}  # 登记允许 PIECEWISE capture 的 token 桶与 LoRA 组合。
def descriptor_key(descriptor):  # 定义 FULL 模式使用的完整且稳定 key。
    return (descriptor["num_tokens"], descriptor["num_reqs"], descriptor["uniform"], descriptor["has_lora"])  # 返回会影响全图复用安全性的字段元组。
def dispatch(descriptor, configured_mode="FULL_AND_PIECEWISE"):  # 按配置、兼容性与已登记 key 选择实际运行模式。
    if descriptor["num_tokens"] is None or descriptor["has_lora"]: return "NONE"  # 对超桶或未捕获 LoRA 组合直接安全降级 eager。
    if descriptor["cascade"]: return "PIECEWISE" if (descriptor["num_tokens"], False) in piecewise_keys and "PIECEWISE" in configured_mode else "NONE"  # 对 cascade 禁止 FULL 并尝试 piecewise。
    if "FULL" in configured_mode and descriptor_key(descriptor) in full_keys: return "FULL"  # 在完整 key 已登记时优先选择全图模式。
    if "PIECEWISE" in configured_mode and (descriptor["num_tokens"], descriptor["has_lora"]) in piecewise_keys: return "PIECEWISE"  # 未命中全图时尝试分段图模式。
    return "NONE"  # 对所有未知或不兼容组合统一回退 eager。
assert dispatch(decode_batch) == "FULL"  # 校验 uniform decode 优先命中 FULL。
assert dispatch(mixed_batch) == "PIECEWISE"  # 校验非 uniform 批次转入 PIECEWISE。
assert dispatch(oversize_batch) == "NONE"  # 校验超大动态 shape 转入 eager。
assert dispatch(decode_batch, "PIECEWISE") == "PIECEWISE"  # 校验单模式配置不会偷偷走 FULL。
assert descriptor_key(decode_batch) in full_keys  # 校验全图决策确实有已登记 key 支撑。

## 4. 静态缓冲不是普通缓存

捕获图会引用输入张量的设备地址，因此 replay 前通常把新请求数据复制进固定地址的 staging buffer，而不是用新对象替换缓冲。缓冲容量对应 padding 后的 capture size；未使用区域必须按模型合同清零或填充，并配合长度、slot mapping、position 等元数据屏蔽，避免 padding 污染结果。

下面用 Python 列表的对象身份模拟地址稳定。它不模拟 GPU allocator，但能暴露最常见的控制面错误：重放前重新分配对象、输入越界、旧 padding 未清理。

In [ ]:
class StaticBufferPool:  # 定义按图 key 持有固定对象的教学版静态缓冲池。
    def __init__(self): self.buffers = {}  # 初始化从图 key 到固定列表对象的映射。
    def stage(self, graph_key, capacity, values):  # 把本轮动态值复制进固定容量缓冲而不替换对象。
        if len(values) > capacity: return False  # 在实际输入超过容量时拒绝写入并交由上层 fallback。
        buffer = self.buffers.setdefault(graph_key, [0] * capacity)  # 首次为 key 分配对象，之后始终复用同一对象。
        buffer[:] = [0] * capacity  # 在写入前清理旧 token，避免上一轮 padding 残留。
        buffer[:len(values)] = values  # 原地复制真实输入以保持对象地址稳定。
        return True  # 告知 wrapper 本轮 staging 满足容量合同。
buffer_pool = StaticBufferPool()  # 创建供捕获与重放共享的静态缓冲池。
demo_key = ("FULL", descriptor_key(decode_batch))  # 构造全图 wrapper 使用的唯一缓存 key。
assert buffer_pool.stage(demo_key, 4, [7, 8, 9, 10])  # 校验首次输入能写入四 token 静态缓冲。
stable_address = id(buffer_pool.buffers[demo_key])  # 记录首次分配对象身份以模拟设备地址。
assert buffer_pool.stage(demo_key, 4, [5, 6])  # 校验较短下一批可以写入同一 capture 桶。
assert id(buffer_pool.buffers[demo_key]) == stable_address  # 校验 replay 前后缓冲对象身份保持不变。
assert buffer_pool.buffers[demo_key] == [5, 6, 0, 0]  # 校验真实值被原地复制且剩余 padding 已清零。

## 5. Wrapper 的 capture/replay 状态机

dispatcher 先决定 mode 与最终 descriptor，wrapper 再读取 forward context：NONE 或 mode 不匹配就直接调用原函数；mode 匹配时，缓存里无图则 capture，有图则 replay。真实 serving 通常会在 warm-up 阶段完成大部分捕获，以免首请求承担高延迟，但运行时仍要清楚地区分首次捕获、命中重放与 eager。

状态机事件必须进入指标和日志，至少包含 mode、descriptor key、capture/replay/eager 原因以及模型版本。否则动态请求频繁 fallback 时，只看总体吞吐很难定位问题。

In [ ]:
class GraphWrapper:  # 定义同时模拟图缓存、capture、replay 与 eager 的包装器。
    def __init__(self, pool): self.pool, self.graphs, self.events = pool, {}, []  # 初始化静态缓冲引用、图缓存与审计事件。
    def run(self, mode, descriptor, values):  # 根据 dispatcher 决策执行一次教学版 forward。
        if mode == "NONE": self.events.append(("eager", "dispatch_fallback")); return sum(values)  # 对 NONE 模式直接运行 reference 并记录原因。
        graph_key = (mode, descriptor_key(descriptor))  # 将具体运行模式与最终描述符合并成图缓存 key。
        if not self.pool.stage(graph_key, descriptor["num_tokens"], values): self.events.append(("eager", "buffer_overflow")); return sum(values)  # 对容量不匹配显式回退而不错误 replay。
        event = "replay" if graph_key in self.graphs else "capture"  # 根据图缓存是否已有 key 决定捕获或重放。
        self.graphs.setdefault(graph_key, {"capacity": descriptor["num_tokens"]})  # 首次捕获时登记固定容量，命中时保留原图对象。
        self.events.append((event, graph_key))  # 记录可用于命中率与首请求延迟分析的事件。
        return sum(self.pool.buffers[graph_key][:descriptor["actual_tokens"]])  # 只读取真实 token 区域以模拟 padding mask。
wrapper = GraphWrapper(StaticBufferPool())  # 创建隔离的 wrapper 供状态转换测试。
assert wrapper.run("FULL", decode_batch, [1, 2, 3, 4]) == 10  # 校验首次全图路径与 eager 求和结果一致。
assert wrapper.events[-1][0] == "capture"  # 校验合法 key 首次执行进入 capture 状态。
assert wrapper.run("FULL", decode_batch, [4, 3, 2, 1]) == 10  # 校验同一 key 的第二批数据可正确执行。
assert wrapper.events[-1][0] == "replay"  # 校验同一图 key 的第二次执行进入 replay 状态。
assert len(wrapper.graphs) == 1  # 校验重复请求没有创建冗余图对象。

## 6. 动态 shape 与不兼容特性如何 fallback

Fallback 必须是正常控制流，不应被当作异常吞掉。典型原因包括：token 数超过所有 capture size、descriptor key 尚未捕获、后端只支持 uniform decode、启用了未纳入 key 的 LoRA、cascade attention、模型热更新导致版本变化，或运行时元数据地址不稳定。

策略应先保证正确性，再谈命中率。可以通过扩充热点桶、离线 warm-up 或把不兼容算子切到 piecewise 提高覆盖，但不能把未知 shape 强塞进“最接近”的图。每类 fallback 都要带 reason code，便于区分正常长尾与配置回归。

In [ ]:
cascade_batch = make_descriptor([3, 2], cascade=True)  # 构造不能使用完整图但可使用分段图的 cascade 批次。
lora_batch = make_descriptor([1, 1, 1, 1], has_lora=True)  # 构造本教学 key 集合尚未捕获的 LoRA 批次。
unknown_full_key_batch = make_descriptor([1, 1, 1])  # 构造 token 桶存在但完整描述符未登记的三请求批次。
assert dispatch(cascade_batch) == "PIECEWISE"  # 校验 cascade 被强制绕开 FULL 并走分段图。
assert dispatch(lora_batch) == "NONE"  # 校验未登记 LoRA 组合安全回退 eager。
assert dispatch(unknown_full_key_batch) == "PIECEWISE"  # 校验 FULL key 缺失时仍可按优先级尝试分段图。
assert wrapper.run("NONE", oversize_batch, list(range(18))) == sum(range(18))  # 校验动态超桶请求回退后仍保持 reference 正确性。

## 7. 配置模式还要受 attention backend 能力约束

配置想要 FULL，不等于注意力后端一定支持。设计文档把能力从强到弱区分为 `ALWAYS`、`UNIFORM_BATCH`、`UNIFORM_SINGLE_TOKEN_DECODE`、`NEVER`；混合后端模型应取最弱能力。初始化阶段据此把用户配置解析为可执行配置，例如 FULL 不可用但 piecewise 可用时降级为双模式或 PIECEWISE。

解析后的配置、降级原因与最终合法 key 集合应固定在模型实例版本上。热切换 backend、量化方式或模型权重时，要重建并预热图缓存，不能沿用旧图。

In [ ]:
support_rank = {"NEVER": 0, "UNIFORM_SINGLE_TOKEN_DECODE": 1, "UNIFORM_BATCH": 2, "ALWAYS": 3}  # 定义注意力后端的 CUDA Graph 能力顺序。
def resolve_mode(requested, backend_support, piecewise_enabled=True):  # 根据最弱后端能力和编译条件解析安全配置。
    if requested == "PIECEWISE" and not piecewise_enabled: return "NONE"  # 在未启用分段编译时禁止 PIECEWISE。
    if requested == "FULL" and support_rank[backend_support] < 3: return "FULL_AND_PIECEWISE" if piecewise_enabled and support_rank[backend_support] > 0 else ("PIECEWISE" if piecewise_enabled else "NONE")  # 对不完全支持 FULL 的后端选择最接近安全模式。
    if requested == "FULL_AND_PIECEWISE" and support_rank[backend_support] == 0: return "PIECEWISE" if piecewise_enabled else "NONE"  # 对完全不支持全图的后端保留可用的分段图路径。
    return requested  # 对满足能力合同的配置保持用户选择。
assert resolve_mode("FULL", "ALWAYS") == "FULL"  # 校验能力最强后端可保留 FULL 配置。
assert resolve_mode("FULL", "UNIFORM_BATCH") == "FULL_AND_PIECEWISE"  # 校验只支持统一批次的后端启用双模式降级。
assert resolve_mode("FULL_AND_PIECEWISE", "NEVER") == "PIECEWISE"  # 校验全图完全不可用时仍可保留分段图。
assert resolve_mode("PIECEWISE", "ALWAYS", False) == "NONE"  # 校验缺少 piecewise compilation 时只能 eager。

## 8. 验收不能只看平均吞吐

上线验收至少覆盖四组指标。正确性上，对每个 capture key 比较 eager 与 graph 输出，并覆盖 padding、边界桶、spec-decode、LoRA、cascade 和超长输入。性能上分开报告 capture latency、replay latency、TTFT、TPOT、吞吐和不同并发下的 P50/P95/P99。资源上统计静态缓冲与图对象显存、启动预热时间。稳定性上统计 FULL/PIECEWISE/eager 占比、每种 fallback 原因与新 key 抖动。

还要设置发布门禁：正确性差异超阈值立即阻断；fallback 激增触发告警但不牺牲正确性；显存余量不足时减少 capture size；模型或 backend 版本变化时失效旧图。

In [ ]:
acceptance_batches = [decode_batch, mixed_batch, cascade_batch, lora_batch, oversize_batch, decode_batch]  # 组织覆盖全图、分段图与 eager 的小型验收流量。
selected_modes = [dispatch(batch) for batch in acceptance_batches]  # 对每个批次执行集中式调度并收集实际模式。
mode_counts = {mode: selected_modes.count(mode) for mode in runtime_modes}  # 汇总三个运行模式的命中次数。
fallback_rate = mode_counts["NONE"] / len(selected_modes)  # 计算 eager fallback 在验收流量中的比例。
graph_coverage = (mode_counts["FULL"] + mode_counts["PIECEWISE"]) / len(selected_modes)  # 计算任一 CUDA Graph 路径的覆盖率。
assert selected_modes == ["FULL", "PIECEWISE", "PIECEWISE", "NONE", "NONE", "FULL"]  # 校验验收样本的逐项路由符合设计合同。
assert sum(mode_counts.values()) == len(acceptance_batches)  # 校验模式计数没有遗漏或重复请求。
assert fallback_rate == 2 / 6  # 校验动态 shape 与 LoRA 两类请求被明确计入 fallback。
assert graph_coverage == 4 / 6  # 校验图覆盖率同时包含 FULL 与 PIECEWISE 命中。
assert wrapper.run("NONE", mixed_batch, [2, 3, 4, 5, 6]) == sum([2, 3, 4, 5, 6])  # 校验 eager reference 可作为图路径正确性基线。

## 面试总结

一个完整回答应按以下链路展开：scheduler 产生真实请求元数据 → 构造最小充分的 `BatchDescriptor` → dispatcher 查询版本化合法 key，并按 FULL、PIECEWISE、NONE 选择 → wrapper 对固定地址缓冲做原地 staging → 首次合法 key capture、后续 replay → 未知 shape 或不兼容能力带 reason code 回退 eager。

最后主动补充三个取舍：更多 capture size 提高覆盖却增加显存和预热；FULL 往往降低小批次延迟却比 PIECEWISE 更挑动态控制流；eager fallback 是安全阀而不是失败。生产上线必须用真实模型、GPU、驱动、后端和线上 shape 分布复测数值一致性、尾延迟、图命中率、显存峰值与版本切换，不能把本 Notebook 的列表状态机当作 CUDA 性能结论。